# **EXPERIMENT-6** (RAG)

Experiment the response of LLM model with and without RAG with different vector data store and record your observation


SHRIHARIHARAN S

24MCS1058

In [12]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
import faiss
import torch


MODEL_NAME = "distilgpt2"
EMBEDDING_MODEL = "paraphrase-MiniLM-L3-v2"

# Check for GPU availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Initialize models with optimization
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
embedding_model = SentenceTransformer(EMBEDDING_MODEL).to(device)

# Pre-process documents (do this once)
documents = [
    "Glioma is a type of brain tumor.",
    "MRI is commonly used for brain imaging.",
    "Sweden's population is approximately 10.4 million as of 2025.",
    "Pinecone is a managed vector database for machine learning applications.",
    "FAISS is a library for efficient similarity search on dense vectors."
]

# Generate and index document embeddings (do this once)
doc_embeddings = embedding_model.encode(documents)
index = faiss.IndexFlatL2(doc_embeddings.shape[1])
index.add(doc_embeddings)

# Function to clean model outputs
def clean_output(text, query, context=None):
    # Remove repetitions of the question
    text = text.replace(f"Question: {query}", "")

    # Remove "A:" and similar prefixes
    text = text.replace("A:", "").replace("Answer:", "")

    # Remove context repetition if context is provided
    if context:
        text = text.replace(f"Context: {context}", "")

    return text.strip()

# Function to process a single query
def process_query(query):
    # Generate query embedding
    query_embedding = embedding_model.encode([query])

    # Get relevant documents
    k = min(2, len(documents))  # Use fewer documents for context
    distances, indices = index.search(query_embedding, k)
    context = " ".join([documents[idx] for idx in indices[0]])

    # Run inference without RAG
    inputs_no_rag = tokenizer(f"Answer this question: {query}\n\nAnswer:", return_tensors="pt").to(device)
    with torch.no_grad():
        outputs_no_rag = model.generate(
            inputs_no_rag.input_ids,
            max_new_tokens=50,
            pad_token_id=tokenizer.eos_token_id,
            temperature=0.7,
            do_sample=True,
            no_repeat_ngram_size=3
        )
    answer_no_rag = clean_output(tokenizer.decode(outputs_no_rag[0], skip_special_tokens=True), query)

    # Run inference with RAG
    inputs_rag = tokenizer(
        f"Use the following information to answer the question.\n\nInformation:\n{context}\n\nQuestion: {query}\n\nAnswer:",
        return_tensors="pt"
    ).to(device)
    with torch.no_grad():
        outputs_rag = model.generate(
            inputs_rag.input_ids,
            max_new_tokens=50,
            pad_token_id=tokenizer.eos_token_id,
            temperature=0.7,
            do_sample=True,
            no_repeat_ngram_size=3
        )
    answer_rag = clean_output(tokenizer.decode(outputs_rag[0], skip_special_tokens=True), query, context)

    return {
        "query": query,
        "without_rag": answer_no_rag,
        "with_rag": answer_rag
    }

# Process queries
queries = [
    "What is glioma?",
    "What imaging techniques are used for the brain?",
    "What is the population of Sweden?",
    "Explain Pinecone.",
    "What is FAISS used for?"
]

# Run queries sequentially
results = []
for query in queries:
    try:
        result = process_query(query)
        results.append(result)
        print(f"Processed query: {query}")
    except Exception as e:
        print(f"Error processing query '{query}': {e}")

# Display results concisely
for result in results:
    print(f"\nQ: {result['query']}")
    print(f"No RAG: {result['without_rag']}")
    print(f"RAG: {result['with_rag']}")
    print("-" * 40)

# Free up GPU memory
if device == "cuda":
    torch.cuda.empty_cache()

Using device: cuda
Processed query: What is glioma?
Processed query: What imaging techniques are used for the brain?
Processed query: What is the population of Sweden?
Processed query: Explain Pinecone.
Processed query: What is FAISS used for?

Q: What is glioma?
No RAG: Answer this question: What is glioma?

 Glioma is a rare and rare, non-invasive, and highly invasive cancer causing anemia and/or malignancies. Gliomas are not rare and are rare. It is an uncommon disease that is rarely suspected in human patients
RAG: Use the following information to answer the question.

Information:
Glioma is a type of brain tumor. MRI is commonly used for brain imaging.



 Glioma is an abnormal brain tumor that is defined by the age of the brain. It is a combination of the two disorders that are usually associated with gliomas.
Grioma can be characterized by an increased or decreased amount of blood
----------------------------------------

Q: What imaging techniques are used for the brain?
No RAG

**Inference :**



*   We used distilgpt2 as LLM model and paraphrase-MiniLM-L3-v2 as EMBEDDING_MODEL
*  list document which is used as document for RAG

*   Temperature and sampling settings (0.7) provided a balance between deterministic and creative outputs

*  **RAG vs Non-RAG Comparison:** results likely showed that RAG responses contained relevant factual information from your document collection


*  Non-RAG responses were probably more generic and potentially less accurate

*  The quality of retrieved documents heavily influences the quality of RAG outputs



